In [ ]:
%pip install timm==0.9.12 torch torchmetrics torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade tqdm opencv-python pillow --upgrade

In [1]:
import torch
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0)) 

GPU name: NVIDIA GeForce RTX 4060 Ti


In [8]:
# Cell 1
from pathlib import Path
import hashlib, cv2, random, time
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
from tqdm import tqdm
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import timm
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast

# ⚙️ SET YOUR DATA ROOT
DATA_ROOT = Path(r"G:/My Drive/CLPD-MF-Dataset")  
assert DATA_ROOT.exists(), f"Dataset folder not found at {DATA_ROOT}"
print("Found dataset root:", DATA_ROOT)


c:\Users\Mohamed Hazem\anaconda3\envs\dlclass\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found dataset root: G:\My Drive\CLPD-MF-Dataset


In [3]:
def list_images_and_labels(root):
    """
    List all images with labels and magnifications from the new folder structure.
    
    Structure:
    G:/My Drive/CLPD-MF-Dataset/
    ├── MF/
    │   └── Patient Name/
    │       ├── x5/
    │       ├── x10/
    │       └── x20/
    └── Non-MF/
        ├── B cell Lymphoma/
        │   └── Patient Name/
        │       ├── x5/
        │       ├── x10/
        │       └── x20/
        ├── PLEVA-PLC/
        └── pseudolymphoma/
    """
    rows = []
    root = Path(root)
    
    # Process MF folder
    mf_dir = root / "MF"
    if mf_dir.exists():
        for patient_dir in mf_dir.iterdir():
            if not patient_dir.is_dir(): continue
            
            patient_name = patient_dir.name
            # Look for x10 and x20 subfolders
            for mag in ['x10', 'x20']:
                mag_dir = patient_dir / mag
                if mag_dir.exists() and mag_dir.is_dir():
                    # Find all .tif images in this magnification folder
                    for img_path in mag_dir.glob('*.tif'):
                        rows.append({
                            'path': img_path,
                            'label': 'MF',
                            'patient': patient_name,
                            'mag': mag,
                            'subtype': None  # MF has no subtype
                        })
    
    # Process Non-MF folder with subtypes
    nonmf_dir = root / "Non-MF"
    if nonmf_dir.exists():
        # Each subfolder is a disease subtype
        for subtype_dir in nonmf_dir.iterdir():
            if not subtype_dir.is_dir(): continue
            
            subtype = subtype_dir.name  # B cell Lymphoma, PLEVA-PLC, or pseudolymphoma
            
            # Each patient within the subtype
            for patient_dir in subtype_dir.iterdir():
                if not patient_dir.is_dir(): continue
                
                patient_name = patient_dir.name
                
                # Look for x10 and x20 subfolders
                for mag in ['x10', 'x20']:
                    mag_dir = patient_dir / mag
                    if mag_dir.exists() and mag_dir.is_dir():
                        # Find all .tif images in this magnification folder
                        for img_path in mag_dir.glob('*.tif'):
                            rows.append({
                                'path': img_path,
                                'label': 'Non-MF',
                                'patient': patient_name,
                                'mag': mag,
                                'subtype': subtype
                            })
    
    return rows

# Load all images
print("\n" + "="*80)
print("LOADING DATASET")
print("="*80 + "\n")

all_images = list_images_and_labels(DATA_ROOT)

print(f"Total images found: {len(all_images)}")
print(f"\nMagnification distribution:")
mag_counts = Counter([r['mag'] for r in all_images])
for mag, count in sorted(mag_counts.items()):
    print(f"  {mag}: {count} images")

print(f"\nLabel distribution:")
label_counts = Counter([r['label'] for r in all_images])
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count} images")

# Count unique patients
unique_patients = len(set(r['patient'] for r in all_images))
mf_patients = len(set(r['patient'] for r in all_images if r['label'] == 'MF'))
nonmf_patients = len(set(r['patient'] for r in all_images if r['label'] == 'Non-MF'))
print(f"\nUnique patients:")
print(f"  Total: {unique_patients}")
print(f"  MF: {mf_patients}")
print(f"  Non-MF: {nonmf_patients}")

# Show Non-MF subtypes distribution
print(f"\nNon-MF subtypes:")
nonmf_images = [r for r in all_images if r['label'] == 'Non-MF']
subtype_counts = Counter([r['subtype'] for r in nonmf_images])
for subtype, count in sorted(subtype_counts.items()):
    subtype_patients = len(set(r['patient'] for r in nonmf_images if r['subtype'] == subtype))
    print(f"  {subtype}: {count} images ({subtype_patients} patients)")




LOADING DATASET

Total images found: 1106

Magnification distribution:
  x10: 413 images
  x20: 693 images

Label distribution:
  MF: 460 images
  Non-MF: 646 images

Unique patients:
  Total: 63
  MF: 21
  Non-MF: 42

Non-MF subtypes:
  B cell Lymphoma: 291 images (16 patients)
  PLEVA-PLC: 209 images (16 patients)
  pseudolymphoma: 146 images (10 patients)


In [4]:
# Cell 3
PATCH_CACHE = Path('./patch_cache')
PATCH_CACHE.mkdir(exist_ok=True)

def extract_and_cache_patches(img_path, patch_size=512, stride=256, 
                              min_foreground_ratio=0.05, max_patches_per_image=200):
    key = hashlib.sha1(str(img_path).encode()).hexdigest()
    cache_dir = PATCH_CACHE / key
    if cache_dir.exists() and any(cache_dir.iterdir()):
        return sorted([str(p) for p in cache_dir.glob('*.jpg')])

    cache_dir.mkdir(parents=True, exist_ok=True)
    img = Image.open(img_path).convert('RGB')
    W,H = img.size
    patches = []

    for y in range(0, H-patch_size+1, stride):
        for x in range(0, W-patch_size+1, stride):
            crop = img.crop((x,y,x+patch_size,y+patch_size))
            arr = np.asarray(crop)
            v = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            fg_ratio = (v > 10).mean()
            if fg_ratio < min_foreground_ratio: continue
            fname = cache_dir / f'{x}_{y}.jpg'
            crop.save(fname, quality=90)
            patches.append(str(fname))
            if len(patches) >= max_patches_per_image: break
        if len(patches) >= max_patches_per_image: break
    return patches


In [ ]:
# Cell 4
class MFHistologyDataset(Dataset):
    def __init__(self, rows, mag='x20', mode='train', patching=True, patch_size=512,
                 stride=256, transforms=None, max_patches_per_image=100):
        self.rows = [r for r in rows if (mag is None or r['mag']==mag)]
        self.mode = mode
        self.patching = patching
        self.patch_size = patch_size
        self.stride = stride
        self.max_patches_per_image = max_patches_per_image
        self.transforms = transforms
        labels = sorted(list({r['label'] for r in self.rows}))
        self.label2idx = {lab:i for i,lab in enumerate(labels)}


        self.items = []
        for r in self.rows:
            if self.patching:
                patches = extract_and_cache_patches(r['path'], patch_size=self.patch_size,
                                                    stride=self.stride, max_patches_per_image=self.max_patches_per_image)
                for p in patches:
                    self.items.append({'img': p, 'label': self.label2idx[r['label']], 'source': str(r['path'])})
            else:
                self.items.append({'img': str(r['path']), 'label': self.label2idx[r['label']], 'source': str(r['path'])})
        if len(self.items)==0:
            print("Warning: dataset empty for magnification", mag)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        img = Image.open(it['img']).convert('RGB')
        if self.transforms: img = self.transforms(img)
        return img, it['label'], it['source']


train_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [6]:
# Cell 5
def patient_split_stratified(rows, mag='x20', val_frac=0.15, seed=42):

    # group patients by class
    cls_map = {}
    for r in rows:
        if r['mag'] != mag: continue
        cls_map.setdefault(r['label'], {}).setdefault(r['patient'], []).append(r)

    # Now stratify
    train_rows, val_rows = [], []

    rng = random.Random(seed)

    for cls, patients_dict in cls_map.items():
        patients = list(patients_dict.keys())
        rng.shuffle(patients)

        n = len(patients)
        n_val = max(1, int(n * val_frac))

        val_p = set(patients[:n_val]) 

        for p, rlist in patients_dict.items():
            if p in val_p:  
                val_rows += rlist
            else: 
                train_rows += rlist
    return train_rows, val_rows



# x20
train_rows_20, val_rows_20 = patient_split_stratified(all_images, mag='x20', val_frac=0.15)

train_ds_20 = MFHistologyDataset(train_rows_20, mag='x20', mode='train', patching=True, transforms=train_tf)
val_ds_20   = MFHistologyDataset(val_rows_20, mag='x20', mode='val', patching=True, transforms=val_tf)
print(len(train_ds_20), len(val_ds_20))

# x10
train_rows_10, val_rows_10 = patient_split_stratified(all_images, mag='x10', val_frac=0.15)

train_ds_10 = MFHistologyDataset(train_rows_10, mag='x10', mode='train', patching=True, transforms=train_tf)
val_ds_10   = MFHistologyDataset(val_rows_10, mag='x10', mode='val', patching=True, transforms=val_tf)
print(len(train_ds_10), len(val_ds_10))



32346 5076
19440 2862


In [9]:
# Cell 6

def create_model(model_name='resnet50', pretrained=True, num_classes=2, dropout=0.2):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    return model

# Choose model_name = 'resnet50' or 'swin_base_patch4_window12_384' (Swin) or tf_efficientnet_b2
model_name_10x = 'tf_efficientnet_b2'
model_name_20x = 'tf_efficientnet_b2'
device = torch.device('cuda')

# --- Create separate models for x10 and x20 ---
model_10 = create_model(model_name=model_name_10x, pretrained=True, num_classes=2).to(device)
model_20 = create_model(model_name=model_name_20x, pretrained=True, num_classes=2).to(device)

# --- Compute class weights for each dataset separately ---
def get_class_weights(train_ds):
    counts = {}
    for it in train_ds.items:
        counts[it['label']] = counts.get(it['label'], 0) + 1
    total = sum(counts.values())
    weights = [total/counts.get(i,1) for i in range(len(counts))]
    return torch.tensor(weights, dtype=torch.float).to(device)

weights_10 = get_class_weights(train_ds_10)
weights_20 = get_class_weights(train_ds_20)

# --- Loss functions ---
criterion_10 = nn.CrossEntropyLoss(weight=weights_10)
criterion_20 = nn.CrossEntropyLoss(weight=weights_20)

# --- Optimizers ---
optimizer_10 = optim.AdamW(model_10.parameters(), lr=3e-4, weight_decay=1e-4)
optimizer_20 = optim.AdamW(model_20.parameters(), lr=3e-4, weight_decay=1e-4)

# --- GradScalers for mixed precision ---
scaler_10 = torch.amp.GradScaler('cuda')
scaler_20 = torch.amp.GradScaler('cuda')


In [10]:
# Cell 7

# Save Path
save_path = Path("C:/Users/Mohamed Hazem/Graduation Project/Dr. Rushdy/CLPD Dr. Kariman/Mycosis-Fungoides-Classifier/Trained Models")

def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0
    for imgs, labels, _src in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return running_loss/total, correct/total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0
    all_preds=[]
    all_labels=[]
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    acc = correct/total if total>0 else 0
    return running_loss/total if total>0 else 0, acc, torch.cat(all_preds) if all_preds else torch.tensor([]), torch.cat(all_labels) if all_labels else torch.tensor([])

# dataloaders
train_loader_10 = DataLoader(train_ds_10, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader_10   = DataLoader(val_ds_10, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
train_loader_20 = DataLoader(train_ds_20, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader_20   = DataLoader(val_ds_20, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)

# Create separate models
model_10 = create_model(model_name=model_name_10x, pretrained=True, num_classes=2).to(device)
model_20 = create_model(model_name=model_name_20x, pretrained=True, num_classes=2).to(device)

# Separate optimizers and scalers
optimizer_10 = optim.AdamW(model_10.parameters(), lr=3e-4, weight_decay=1e-4)
optimizer_20 = optim.AdamW(model_20.parameters(), lr=3e-4, weight_decay=1e-4)
scaler_10 = torch.amp.GradScaler('cuda')
scaler_20 = torch.amp.GradScaler('cuda')

# Separate best validation accuracy trackers
best_val_acc_10 = 0.0
best_val_acc_20 = 0.0

EPOCHS = 2

for epoch in range(EPOCHS):
    t0 = time.time()

    # --- Train x10 model ---
    train_loss_10, train_acc_10 = train_one_epoch(model_10, train_loader_10, optimizer_10, criterion_10, device, scaler_10)
    val_loss_10, val_acc_10, _, _ = validate(model_10, val_loader_10, criterion_10, device)

    # --- Train x20 model ---
    train_loss_20, train_acc_20 = train_one_epoch(model_20, train_loader_20, optimizer_20, criterion_20, device, scaler_20)
    val_loss_20, val_acc_20, _, _ = validate(model_20, val_loader_20, criterion_20, device)

    t1 = time.time()
    print(f"Epoch {epoch+1}/{EPOCHS} | x10 train_acc {train_acc_10:.4f} val_acc {val_acc_10:.4f} | "
          f"x20 train_acc {train_acc_20:.4f} val_acc {val_acc_20:.4f} | time {(t1-t0):.1f}s")

    # Save best models
    if val_acc_10 > best_val_acc_10:
        best_val_acc_10 = val_acc_10
        torch.save(model_10.state_dict(), save_path / f'model_{model_name_10x}_x10.pth')
        print("Saved best x10 model.")
    if val_acc_20 > best_val_acc_20:
        best_val_acc_20 = val_acc_20
        torch.save(model_20.state_dict(), save_path / f'model_{model_name_20x}_x20.pth')
        print("Saved best x20 model.")



C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_23264\1144052622.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/2 | x10 train_acc 0.8331 val_acc 0.8103 | x20 train_acc 0.8738 val_acc 0.7782 | time 1867.3s
Saved best x10 model.
Saved best x20 model.
Epoch 2/2 | x10 train_acc 0.9093 val_acc 0.8295 | x20 train_acc 0.9338 val_acc 0.7394 | time 1514.9s
Saved best x10 model.


In [9]:
# Cell 8
# --- Load best models ---
model_10.load_state_dict(torch.load(f"G:/My Drive/CLPD-MF-Dataset/Local Models/model_{model_name_10x}_x10.pth"))
model_10.to('cuda').eval()
model_20.load_state_dict(torch.load(f"G:/My Drive/CLPD-MF-Dataset/Local Models/model_{model_name_20x}_x20.pth"))
model_20.to('cuda').eval()

# --- Prediction per model ---
def predict_image_by_patches(img_path, model, device, patch_size=512, stride=256, transforms=val_tf):
    """
    Predicts class for one image by extracting patches.
    Returns mean probability and predicted class.
    """
    patches = extract_and_cache_patches(img_path, patch_size=patch_size, stride=stride, max_patches_per_image=200)
    if len(patches) == 0:
        return None
    probs = []
    with torch.no_grad():
        for p in patches:
            x = transforms(Image.open(p).convert('RGB')).unsqueeze(0).to(device)
            out = model(x)
            prob = torch.softmax(out, dim=1).cpu().numpy()[0]
            probs.append(prob)
    mean_prob = np.array(probs).mean(axis=0)
    pred_class = mean_prob.argmax()
    return {'pred': int(pred_class), 'prob': mean_prob.tolist(), 'num_patches': len(patches)}

# --- Combined prediction for x10 + x20 ---
def combined_prediction(img_paths_dict, model_10, model_20, device):
    """
    img_paths_dict: {'x10': Path(...), 'x20': Path(...)}
    Returns combined probability and predicted class.
    """
    probs = []
    if 'x10' in img_paths_dict:
        res10 = predict_image_by_patches(img_paths_dict['x10'], model_10, device)
        if res10:
            probs.append(res10['prob'])
    if 'x20' in img_paths_dict:
        res20 = predict_image_by_patches(img_paths_dict['x20'], model_20, device)
        if res20:
            probs.append(res20['prob'])
    if len(probs) == 0:
        return None
    combined_prob = np.mean(probs, axis=0)
    combined_class = combined_prob.argmax()
    return {'pred': int(combined_class), 'prob': combined_prob.tolist()}

# --- Example usage ---
example_imgs = {
    'x10': Path("G:/My Drive/CLPD-MF-Dataset/MF/doaa sayed 113-7-21/113-7-21 x10 B tif.tif"),
    'x20': Path("G:/My Drive/CLPD-MF-Dataset/MF/doaa sayed 113-7-21/113-7-21 x20 H .tif")
}
res = combined_prediction(example_imgs, model_10, model_20, device)
label_map = {0: "MF", 1: "Non-MF"}
print(f"Combined prediction: {label_map[res['pred']]} " 
       f"({res['prob'][res['pred']]*100:.1f}% confidence)")

C:\Users\Mohamed Hazem\AppData\Local\Temp\ipykernel_20332\1000918090.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_10.load_state_dict(torch.load(f"G:/My Drive/CL

Combined prediction: MF (65.2% confidence)
